# Webscraping ______________________

In [1]:
### imports
import requests
import io
import pandas as pd
import re

from bs4 import BeautifulSoup as bs

In [2]:
### config
URL = "https://www.billboard.com/charts/hot-100/"

In [3]:
### request url and let bs work
response = requests.get(URL)
soup = bs(response.text, "html.parser")

# cant print huge stuff in notebook w/o changing settings
#print(soup.prettify())

In [4]:
### inspect source and grab what you think holds a song row
cards = soup.select("ul.o-chart-results-list-row")
### check that there is 100 songs/rows/cards
print(len(cards))

100


In [5]:
### run thru cards to see if we like 
for card in cards:
    print(card.get_text(" | ", strip=True).split(" | "))
    print("----------")

['1', "Choosin' Texas", 'Ella Langley', 'LW', '2', 'PEAK', '1', 'WEEKS', '24', 'LW', '2', 'PEAK', '1', 'WEEKS', '24']
----------
['2', 'Swim', 'BTS', 'LW', '1', 'PEAK', '1', 'WEEKS', '2', 'LW', '1', 'PEAK', '1', 'WEEKS', '2']
----------
['3', 'Man I Need', 'Olivia Dean', 'LW', '3', 'PEAK', '2', 'WEEKS', '32', 'LW', '3', 'PEAK', '2', 'WEEKS', '32']
----------
['4', 'I Just Might', 'Bruno Mars', 'LW', '4', 'PEAK', '1', 'WEEKS', '12', 'LW', '4', 'PEAK', '1', 'WEEKS', '12']
----------
['5', 'Ordinary', 'Alex Warren', 'LW', '5', 'PEAK', '1', 'WEEKS', '59', 'LW', '5', 'PEAK', '1', 'WEEKS', '59']
----------
['6', 'So Easy (To Fall In Love)', 'Olivia Dean', 'LW', '7', 'PEAK', '6', 'WEEKS', '27', 'LW', '7', 'PEAK', '6', 'WEEKS', '27']
----------
['7', 'Golden', 'HUNTR/X: EJAE, Audrey Nuna & REI AMI', 'LW', '6', 'PEAK', '1', 'WEEKS', '41', 'LW', '6', 'PEAK', '1', 'WEEKS', '41']
----------
['8', 'Stateside', 'PinkPantheress', 'With', 'Zara Larsson', 'LW', '8', 'PEAK', '6', 'WEEKS', '13', 'LW', '8

In [6]:
### run thru the cards/rows now that we know its structure and create df
rows = []

### for song_details in song_container
for card in cards:
    parts = card.get_text(" | ", strip=True).split(" | ")

    rank = parts[0]
    lw = parts[parts.index("LW") + 1]
    peak = parts[parts.index("PEAK") + 1]
    weeks = parts[parts.index("WEEKS") + 1]

    title_artist_parts = parts[1:parts.index("LW")]
    title_artist_parts = [p for p in title_artist_parts if p != "NEW"]
    title = title_artist_parts[0]
    artist = " ".join(title_artist_parts[1:])

    rows.append({
        "Rank": rank,
        "Title": title,
        "Artist": artist,
        "LW": lw,
        "Peak": peak,
        "Weeks": weeks
    })

top100_df = pd.DataFrame(rows)
top100_df

,Rank,Title,Artist,LW,Peak,Weeks
0,1,Choosin' Texas,Ella Langley,2,1,24
1,2,Swim,BTS,1,1,2
2,3,Man I Need,Olivia Dean,3,2,32
3,4,I Just Might,Bruno Mars,4,1,12
4,5,Ordinary,Alex Warren,5,1,59
...,...,...,...,...,...,...
95,96,Merry Go Round,BTS,52,52,2
96,97,Griddle,Yeat & Don Toliver,-,97,1
97,98,Silent Treatment,Freya Skye,-,98,1
98,99,Taste Back,Harry Styles,69,17,4


# extract out the This Week, artist, song, Last Week, Peak Position, and Weeks on Chart values into a pandas DataFrame

In [7]:
### fetch song titles[]
# clean = re.sub(r'[\t\n]+', '', text)
song_titles = soup.find_all('h3', class_='c-title a-font-basic u-letter-spacing-0010 u-max-width-397 lrv-u-font-size-16 lrv-u-font-size-14@mobile-max u-line-height-22px u-word-spacing-0063 u-line-height-normal@mobile-max a-truncate-ellipsis-2line lrv-u-margin-b-025 lrv-u-margin-b-00@mobile-max')
song_titles = [(re.sub(r'[\t\n]+', '', x.text)) for x in song_titles]
song_titles

["Choosin' Texas",
 'Swim',
 'Man I Need',
 'I Just Might',
 'Ordinary',
 'So Easy (To Fall In Love)',
 'Golden',
 'Stateside',
 'The Fate Of Ophelia',
 'Folded',
 'Where Is My Husband!',
 'Be Her',
 'Opalite',
 'Sleepless In A Hotel Room',
 'Risk It All',
 'Babydoll',
 'Iloveitiloveitiloveit',
 'Dracula',
 'E85',
 'Yukon',
 'Father',
 'Be By You',
 'American Girls',
 'DTMF',
 'Homewrecker',
 'Body',
 'All The Love',
 'The Fall',
 'Die On This Hill',
 'Change My Mind',
 'Pop Dat Thang',
 'White Keys',
 'Fever Dream',
 "Let 'Em Know",
 'Elizabeth Taylor',
 'Lush Life',
 'What You Saying',
 'The Great Divide',
 'Days Like These',
 'King',
 'Motion Party',
 'Body To Body',
 'Midnight Sun',
 'Punch Drunk',
 'Brunette',
 'Loving Life Again',
 'McArthur',
 'Porch Light',
 'Whatever Works',
 'When Did You Get Hot?',
 'This A Must',
 'Raindance',
 'Chanel',
 'Dandelion',
 'Rethink Some Things',
 "I Ain't Coming Back",
 'Preacher Man',
 'Aperture',
 "Mama's Favorite",
 'In My Room',
 'Bully',
 

In [8]:
### fetch artists[]
# clean = re.sub(r'[\t\n]+', '', text)

# <span class="
#     c-label a-no-trucate
#     a-font-secondary u-font-size-15 u-font-size-13@mobile-max u-line-height-18px@mobile-max u-letter-spacing-0010 u-line-height-21px
#     a-children-link-color-black a-children-link-color-brand-secondary:hover
#     lrv-a-children-link-decoration-underline:hover
#     lrv-u-display-block a-truncate-ellipsis-2line u-max-width-397 u-max-width-230@tablet-only u-max-width-300@mobile-max">
# 							<a href="https://www.billboard.com/artist/bts/">BTS</a>						</span>


artists = soup.find_all('span', class_='c-label a-no-trucate a-font-secondary u-font-size-15 u-font-size-13@mobile-max u-line-height-18px@mobile-max u-letter-spacing-0010 u-line-height-21px a-children-link-color-black a-children-link-color-brand-secondary:hover lrv-a-children-link-decoration-underline:hover lrv-u-display-block a-truncate-ellipsis-2line u-max-width-397 u-max-width-230@tablet-only u-max-width-300@mobile-max')
artists = [(re.sub(r'[\t\n]+', '', x.text)).strip() for x in artists]
artists

['Ella Langley',
 'BTS',
 'Olivia Dean',
 'Bruno Mars',
 'Alex Warren',
 'Olivia Dean',
 'HUNTR/X: EJAE, Audrey Nuna & REI AMI',
 'PinkPantheress With Zara Larsson',
 'Taylor Swift',
 'Kehlani',
 'RAYE',
 'Ella Langley',
 'Taylor Swift',
 'Luke Combs',
 'Bruno Mars',
 'Dominic Fike',
 'Bella Kay',
 'Tame Impala & JENNIE',
 'Don Toliver',
 'Justin Bieber',
 'Ye & Travis Scott',
 'Luke Combs',
 'Harry Styles',
 'Bad Bunny',
 'sombr',
 'Don Toliver',
 'Ye & Andre Troutman',
 'Cody Johnson',
 'Sienna Spiro',
 'Riley Green',
 'DaBaby',
 'Dominic Fike',
 'Alex Warren',
 'T.I.',
 'Taylor Swift',
 'Zara Larsson',
 'Lil Uzi Vert',
 'Noah Kahan',
 'Luke Combs',
 'Ye',
 'BossMan Dlow',
 'BTS',
 'Zara Larsson',
 'Ye',
 'Tucker Wetmore',
 'Ella Langley',
 'HARDY, Eric Church, Morgan Wallen & Tim McGraw',
 'Noah Kahan',
 'Ye',
 'Sabrina Carpenter',
 'Ye',
 'Dave & Tems',
 'Tyla',
 'Ella Langley',
 'Luke Combs',
 'Morgan Wallen Featuring Post Malone',
 'Ye',
 'Harry Styles',
 'Ye',
 'Julia Wolf',
 'Y

In [9]:
### see if there's a pattern inside the span class
rows = soup.select("li.o-chart-results-list__item")
i=0
for row in rows:
    print("["+str(i)+"]"+row.get_text(" | ", strip=True))
    print("-" * 80)
    i += 1

[0]1
--------------------------------------------------------------------------------
[1]
--------------------------------------------------------------------------------
[2]
--------------------------------------------------------------------------------
[3]Choosin' Texas | Ella Langley
--------------------------------------------------------------------------------
[4]
--------------------------------------------------------------------------------
[5]2
--------------------------------------------------------------------------------
[6]1
--------------------------------------------------------------------------------
[7]24
--------------------------------------------------------------------------------
[8]
--------------------------------------------------------------------------------
[9]2
--------------------------------------------------------------------------------
[10]1
--------------------------------------------------------------------------------
[11]24
---------------------

In [10]:
### for thru span class
billboard = [x.get_text(strip=True) for x in soup.select("li.o-chart-results-list__item")]
billboard = [x for x in billboard if x]
billboard

['1',
 "Choosin' TexasElla Langley",
 '2',
 '1',
 '24',
 '2',
 '1',
 '24',
 '2',
 'SwimBTS',
 '1',
 '1',
 '2',
 '1',
 '1',
 '2',
 '3',
 'Man I NeedOlivia Dean',
 '3',
 '2',
 '32',
 '3',
 '2',
 '32',
 '4',
 'I Just MightBruno Mars',
 '4',
 '1',
 '12',
 '4',
 '1',
 '12',
 '5',
 'OrdinaryAlex Warren',
 '5',
 '1',
 '59',
 '5',
 '1',
 '59',
 '6',
 'So Easy (To Fall In Love)Olivia Dean',
 '7',
 '6',
 '27',
 '7',
 '6',
 '27',
 '7',
 'GoldenHUNTR/X: EJAE, Audrey Nuna & REI AMI',
 '6',
 '1',
 '41',
 '6',
 '1',
 '41',
 '8',
 'StatesidePinkPantheressWithZara Larsson',
 '8',
 '6',
 '13',
 '8',
 '6',
 '13',
 '9',
 'The Fate Of OpheliaTaylor Swift',
 '9',
 '1',
 '26',
 '9',
 '1',
 '26',
 '10',
 'FoldedKehlani',
 '10',
 '6',
 '42',
 '10',
 '6',
 '42',
 '11',
 'Where Is My Husband!RAYE',
 '17',
 '11',
 '27',
 '17',
 '11',
 '27',
 '12',
 'Be HerElla Langley',
 '13',
 '12',
 '7',
 '13',
 '12',
 '7',
 '13',
 'OpaliteTaylor Swift',
 '12',
 '1',
 '26',
 '12',
 '1',
 '26',
 '14',
 'Sleepless In A Hotel Room

In [11]:
### this is when i learned i needed to go a diff route
### when there's a new song on the list. they mess with the indexes

rank = billboard[0::8]
# title_artist = billboard[3::14]
# lw = billboard[5::14]
# peak = billboard[6::14]
# weeks = billboard[7::14]

billboard_df = pd.DataFrame({
    "Rank": rank
    # "Title/Artist": title_artist,
    # "LW": lw,
    # "Peak": peak,
    # "Weeks": weeks
})

billboard_df.head(54)

,Rank
0,1
1,2
2,3
3,4
4,5
5,6
6,7
7,8
8,9
9,10
